In [1]:
import sys; sys.path.append('..')
from malign_logits import *

In [2]:
psyche = Psyche.from_pretrained()

Detected Mac - using MPS (Metal Performance Shaders) with torch.float16
Loading base model...


Loading weights: 100%|██████████| 291/291 [00:03<00:00, 82.33it/s, Materializing param=model.norm.weight]                               


Loading instruct model...


Loading weights: 100%|██████████| 291/291 [00:04<00:00, 60.87it/s, Materializing param=model.norm.weight]                              


Both models loaded.


In [13]:
import pandas as pd

# prompt = "She knelt down in front of him and reached for his"
# prompt = "He put his hands between her"
# prompt = "She was attractive. She had big"
# prompt = "He was so angry he wanted to"
prompt = "She was so angry she wanted to"
# prompt = "He lay naked and his bed and"
# prompt = "She started to suck his"
# prompt = "I'm so tired I want to"
# prompt = "She ran her finger down to his"
# prompt = "Don't make a sound or I'll"
# prompt = "He came inside"
# prompt = "He put his hands around her"
# prompt = "He unzipped his"
# prompt = "She wrapped her lips around the"
# prompt = "She felt him growing"
# prompt = "He reached into his jacket and pulled out a"


s = psyche.analyze(prompt)
dm = s.displacement_map(layers=list(range(1,33)))

  Sublimation axis: 19 sublimated, 12 introduced
  Repression axis: 33 repressed, 3 amplified
  Total unique words to embed: 46


Computing contextual embeddings: 100%|██████████| 32/32 [00:00<00:00, 101.32it/s]

  Loaded 1472 embeddings (1472/1472 expected)


In [14]:
import numpy as np
import torch
from collections import defaultdict

def permutation_test(dm, prompt, psyche, layer=16, n_permutations=1000, min_prob=0.003, delta_threshold=0.003):
    """
    Test whether sublimation pairs are more semantically similar than
    random pairings of sublimated × introduced words.
    
    Returns observed mean similarity, null distribution, and p-value.
    """
    df = dm['df']
    dt = delta_threshold

    sig = df[
        (df['base'] > min_prob) | (df['ego'] > min_prob) | (df['superego'] > min_prob)
    ]

    sublimated = sig[sig['ego - base'] < -dt]['word'].tolist()
    introduced = sig[sig['ego - base'] > dt]['word'].tolist()

    if not sublimated or not introduced:
        print("Not enough words in one or both groups")
        return None

    # Get embeddings (reuse cached if available)
    stash = psyche._stash
    model = psyche.ego.model
    tokenizer = psyche.ego.tokenizer
    device = psyche.ego.device

    def get_emb(word):
        if stash is not None:
            key = ("embedding", prompt, word, layer)
            if key in stash:
                return torch.as_tensor(stash[key], dtype=torch.float32)
        text = prompt + " " + word
        ids = tokenizer.encode(text, return_tensors="pt").to(device)
        prompt_len = len(tokenizer.encode(prompt))
        with torch.no_grad():
            outputs = model(ids, output_hidden_states=True)
            hidden = outputs.hidden_states[layer]
            word_hidden = hidden[0, prompt_len:, :].mean(dim=0).cpu()
        return torch.nn.functional.normalize(
            word_hidden.float().unsqueeze(0), dim=-1
        ).squeeze()

    embs = {}
    for w in sublimated + introduced:
        try:
            embs[w] = get_emb(w)
        except Exception:
            continue

    sublimated = [w for w in sublimated if w in embs]
    introduced = [w for w in introduced if w in embs]

    # Observed: mean similarity between actual displacement pairs
    # (every sublimated word × every introduced word)
    obs_sims = []
    for sw in sublimated:
        for iw in introduced:
            sim = torch.dot(embs[sw], embs[iw]).item()
            obs_sims.append(sim)
    observed_mean = np.mean(obs_sims)

    # Null distribution: shuffle which words are "sublimated" vs "introduced"
    all_words = sublimated + introduced
    n_sub = len(sublimated)
    null_means = []

    for _ in range(n_permutations):
        perm = np.random.permutation(all_words)
        perm_sub = perm[:n_sub].tolist()
        perm_int = perm[n_sub:].tolist()
        sims = []
        for sw in perm_sub:
            for iw in perm_int:
                sims.append(torch.dot(embs[sw], embs[iw]).item())
        null_means.append(np.mean(sims))

    null_means = np.array(null_means)
    p_value = np.mean(null_means >= observed_mean)

    print(f"Layer {layer}")
    print(f"  Sublimated words: {len(sublimated)}, Introduced words: {len(introduced)}")
    print(f"  Observed mean similarity: {observed_mean:.4f}")
    print(f"  Null distribution: mean={null_means.mean():.4f}, std={null_means.std():.4f}")
    print(f"  p-value: {p_value:.4f}")
    print(f"  {'SIGNIFICANT' if p_value < 0.05 else 'NOT SIGNIFICANT'} (p < 0.05)")

    return {
        'observed_mean': observed_mean,
        'null_means': null_means,
        'p_value': p_value,
        'sublimated': sublimated,
        'introduced': introduced,
        'layer': layer,
    }

# Run at multiple layers
results = {}
for layer in [8, 16, 24, 32]:
    results[layer] = permutation_test(dm, prompt, psyche, layer=layer)

# Plot the null distribution vs observed
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=2, subplot_titles=[f'Layer {l}' for l in [8, 16, 24, 32]])

for i, layer in enumerate([8, 16, 24, 32]):
    r = results[layer]
    if r is None:
        continue
    row, col = divmod(i, 2)
    fig.add_trace(go.Histogram(
        x=r['null_means'], nbinsx=50, name=f'null (L{layer})',
        marker_color='rgba(150,150,150,0.6)', showlegend=False,
    ), row=row+1, col=col+1)
    fig.add_vline(
        x=r['observed_mean'], line_color='red', line_width=2,
        annotation_text=f"observed ({r['p_value']:.3f})",
        annotation_font_color='red',
        row=row+1, col=col+1,
    )

fig.update_layout(
    title=f'Permutation test: sublimation displacement — "{prompt}"',
    height=600, width=900, template='plotly_white',
)
fig.show()

Layer 8
  Sublimated words: 19, Introduced words: 12
  Observed mean similarity: 0.5760
  Null distribution: mean=0.5893, std=0.0056
  p-value: 0.9760
  NOT SIGNIFICANT (p < 0.05)
Layer 16
  Sublimated words: 19, Introduced words: 12
  Observed mean similarity: 0.6049
  Null distribution: mean=0.6140, std=0.0046
  p-value: 0.9530
  NOT SIGNIFICANT (p < 0.05)
Layer 24
  Sublimated words: 19, Introduced words: 12
  Observed mean similarity: 0.5400
  Null distribution: mean=0.5533, std=0.0057
  p-value: 0.9730
  NOT SIGNIFICANT (p < 0.05)
Layer 32
  Sublimated words: 19, Introduced words: 12
  Observed mean similarity: 0.5588
  Null distribution: mean=0.5769, std=0.0083
  p-value: 0.9700
  NOT SIGNIFICANT (p < 0.05)


In [15]:
def permutation_test_nn(dm, prompt, psyche, layer=16, n_permutations=1000,
                        min_prob=0.003, delta_threshold=0.003):
    """
    Test whether each sublimated word's BEST match among introduced words
    is more similar than expected under random label assignment.
    """
    df = dm['df']
    dt = delta_threshold
    sig = df[(df['base'] > min_prob) | (df['ego'] > min_prob) | (df['superego'] > min_prob)]

    sublimated = sig[sig['ego - base'] < -dt]['word'].tolist()
    introduced = sig[sig['ego - base'] > dt]['word'].tolist()

    stash = psyche._stash

    def get_emb(word):
        if stash is not None:
            key = ("embedding", prompt, word, layer)
            if key in stash:
                return torch.as_tensor(stash[key], dtype=torch.float32)
        text = prompt + " " + word
        ids = psyche.ego.tokenizer.encode(text, return_tensors="pt").to(psyche.ego.device)
        prompt_len = len(psyche.ego.tokenizer.encode(prompt))
        with torch.no_grad():
            outputs = psyche.ego.model(ids, output_hidden_states=True)
            hidden = outputs.hidden_states[layer]
            word_hidden = hidden[0, prompt_len:, :].mean(dim=0).cpu()
        return torch.nn.functional.normalize(word_hidden.float().unsqueeze(0), dim=-1).squeeze()

    embs = {}
    for w in sublimated + introduced:
        try:
            embs[w] = get_emb(w)
        except:
            continue

    sublimated = [w for w in sublimated if w in embs]
    introduced = [w for w in introduced if w in embs]
    all_words = sublimated + introduced
    n_sub = len(sublimated)

    def mean_best_match(sub_list, int_list):
        best_sims = []
        for sw in sub_list:
            max_sim = max(torch.dot(embs[sw], embs[iw]).item() for iw in int_list)
            best_sims.append(max_sim)
        return np.mean(best_sims)

    observed = mean_best_match(sublimated, introduced)

    null_means = []
    for _ in range(n_permutations):
        perm = np.random.permutation(all_words)
        p_sub = perm[:n_sub].tolist()
        p_int = perm[n_sub:].tolist()
        null_means.append(mean_best_match(p_sub, p_int))

    null_means = np.array(null_means)
    p_value = np.mean(null_means >= observed)

    print(f"Layer {layer} (nearest-neighbor test)")
    print(f"  Observed mean best-match similarity: {observed:.4f}")
    print(f"  Null distribution: mean={null_means.mean():.4f}, std={null_means.std():.4f}")
    print(f"  p-value: {p_value:.4f}")
    print(f"  {'SIGNIFICANT' if p_value < 0.05 else 'NOT SIGNIFICANT'}")

    return {
        'observed': observed, 'null_means': null_means,
        'p_value': p_value, 'layer': layer,
    }

for layer in [8, 16, 24]:
    permutation_test_nn(dm, prompt, psyche, layer=layer)

Layer 8 (nearest-neighbor test)
  Observed mean best-match similarity: 0.6914
  Null distribution: mean=0.7410, std=0.0181
  p-value: 0.9940
  NOT SIGNIFICANT
Layer 16 (nearest-neighbor test)
  Observed mean best-match similarity: 0.7049
  Null distribution: mean=0.7437, std=0.0150
  p-value: 0.9920
  NOT SIGNIFICANT
Layer 24 (nearest-neighbor test)
  Observed mean best-match similarity: 0.6575
  Null distribution: mean=0.7014, std=0.0189
  p-value: 0.9840
  NOT SIGNIFICANT


In [4]:
fig = plot_sublimation(dm, prompt, min_sim=0.75)
fig.show()

In [5]:
# Or top sublimated words
fig = plot_layer_displacement(dm, prompt)

In [ ]:
# !pip install kaleido